# Training & Experiments — Method 1: k-NN

Training and experimentation happen here (NOT in the test notebook `powerpredict.ipynb`).
At the end we save the best model to `models/knn_model.joblib`.

In [ ]:
import os, sys
# Make src/ and src/method_1 importable (notebook runs in notebooks/)
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
sys.path.append(os.path.abspath(os.path.join('..', 'src', 'method_1')))

from preprocessing import load_data
import model as knn  # src/method_1/model.py

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor

In [ ]:
X, y = load_data()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, '| Val:', X_val.shape)

## 1. Baseline (Dummy) — reference value we must beat

In [ ]:
dummy = DummyRegressor().fit(X_train, y_train)
print('Dummy MAE (val):', mean_absolute_error(y_val, dummy.predict(X_val)))

## 2. Tune k systematically (cross-validation, metric = MAE)

In [ ]:
best_model, search = knn.tune(X_train, y_train)
print('Val MAE (best model):', mean_absolute_error(y_val, best_model.predict(X_val)))

In [ ]:
# Full CV results (useful as report material)
import pandas as pd
cv = pd.DataFrame(search.cv_results_)
cv['MAE'] = -cv['mean_test_score']
cv[['param_knn__n_neighbors', 'param_knn__weights', 'MAE']].sort_values('MAE')

## 3. Experiment: encode weather text columns vs. drop them

In [ ]:
for enc in [True, False]:
    m = knn.train(X_train, y_train, n_neighbors=11, encode_categoricals=enc)
    mae = mean_absolute_error(y_val, m.predict(X_val))
    print(f'encode_categoricals={enc}: val MAE = {mae:.2f}')

## 4. Save the best model + check size (must be <= 50 MB)

In [ ]:
# Train on ALL data and save
final = best_model.fit(X, y)
knn.save(final)
size_mb = os.path.getsize(knn.MODEL_PATH) / 1e6
print(f'Saved: {knn.MODEL_PATH} ({size_mb:.1f} MB)')
assert size_mb <= 50, 'Model too large! Try encode_categoricals=False.'